# tenuretrack

**Nobody tells assistant professors the numbers. This notebook computes them.**

Give it your ORCID, your university, and the year your tenure-line appointment
began. It builds a cohort of early-career faculty in your subfield from
[OpenAlex](https://openalex.org), works out what their publication records
looked like at each year of the tenure clock, and shows where your record sits
at the same point on the clock.

You do not need to install anything or know any Python. Run the cells in order
from the top. Each one has a play button on its left.

### What you need

1. Your ORCID (for example `0000-0002-1825-0097`). If you do not have one, get
   one free at [orcid.org](https://orcid.org).
2. Your university's name.
3. The first calendar year of your tenure-line appointment.
4. Your work email address. OpenAlex asks every caller to identify itself, and
   politely identified callers get faster and more reliable service. Your
   address goes to OpenAlex and nowhere else.
5. **A free OpenAlex API key. Get this before you start.** It is the difference
   between finishing in one sitting and coming back for three days. Make an
   account at [openalex.org](https://openalex.org), then copy your key from
   [openalex.org/settings/api](https://openalex.org/settings/api). It takes
   about thirty seconds, costs nothing, and needs no payment details. It cannot
   be charged for anything: when the free allowance runs out OpenAlex refuses
   the request rather than billing you. The notebook asks for it in step 3.

   You can run without one. The next section says what that costs you.

### How long it takes

**With the free key, 10 to 40 minutes**, depending on how large your subfield
is. That is the whole thing, start to finish, in one sitting.

**Without a key, plan on two or three days.** OpenAlex gives an unidentified
caller about 1,000 requests a day, and building a cohort spends a few thousand.
The run stops when the day's allowance is gone and continues when you come back
tomorrow. It works. It is a slow way to spend a week, and thirty seconds spent
on the key at [openalex.org/settings/api](https://openalex.org/settings/api)
skips all of it.

Either way, an interruption costs you nothing. Everything fetched is cached, so
rerunning the cell picks up where it stopped rather than starting over.

### About the people in your cohort

Building the cohort means downloading OpenAlex records for other early-career
faculty in your subfield, and those records have names on them. Those names
stay on this temporary machine. The report never contains a cohort member's
name, ID, or individual numbers, and the report is the only thing you download.
The one person named in your results is you.

### A caution about what these numbers are

They describe what a group of people in your field did. They are not a
standard, and nobody agreed to be measured against them. Read them the way you
would read a distribution of salaries or class sizes: useful context for a
conversation, not a verdict on anyone.

---
## Step 1: Install the tool

Run this once per session. It takes about a minute. Colab gives you a fresh
machine each time, so if you come back tomorrow, run it again.

In [ ]:
#@title Install tenuretrack { display-mode: "form" }
!pip install --quiet "git+https://github.com/sp8rks/tenuretrack.git"

import tenuretrack
print("tenuretrack", tenuretrack.__version__, "is ready.")

---
## Step 2 (optional): Keep your work in Google Drive

Colab throws the machine away when you close the tab. If you connect Google
Drive, the downloaded OpenAlex records and your results are saved there
instead, so a long build can be resumed tomorrow and you keep your report.

Skip this cell if you would rather keep everything on the temporary machine and
just download the report at the end.

Note that if you do connect Drive, the saved cache includes cohort members'
names, in a folder only you can see.

In [ ]:
#@title Connect Google Drive (optional) { display-mode: "form" }
import os
from pathlib import Path

use_drive = True  #@param {type:"boolean"}
folder_name = "tenuretrack"  #@param {type:"string"}

if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    workdir = Path("/content/drive/MyDrive") / folder_name
else:
    workdir = Path("/content") / folder_name

workdir.mkdir(parents=True, exist_ok=True)
os.chdir(workdir)
print("Working in:", workdir)

---
## Step 3: Tell it about you

Fill in the boxes, then run the cell. It looks up your papers on OpenAlex
and proposes the four to six research topics that best describe what you
publish on, with the journals those papers ran in so you can see where each
topic came from.

### If your clock was stopped

`years_the_clock_was_stopped` is for parental leave, medical leave, a pandemic
extension, or anything else that paused your tenure clock. Leave it at 0 if
none applies.

It matters because everything here is a comparison at a point on the clock. If
you are in your sixth calendar year with one year stopped, you are at year five
of the clock, and you should be compared against people at year five rather
than against people who had six uninterrupted years. That is what the extension
is for.

Your papers from all six calendar years still count. An extension gives you
time back; it does not un-write what you published. Your report will say so, in
those words, so that anyone reading it can see the arrangement rather than
guess at it.

The number stays on this machine and in your own report. It is not sent to
OpenAlex, and nothing about why the clock stopped is asked for or recorded.

### The OpenAlex key goes in here

**The cell will ask for the key when you run it.** This is the free key from
item 5 at the top of the notebook. If you have not got one yet, now is the
moment: [openalex.org/settings/api](https://openalex.org/settings/api), about
thirty seconds, and it turns a three-day run into a forty-minute one.

You can skip the key by pressing Enter. The tool still runs, in the slower
shape described at the top, and loses nothing to the stops along the way.

There is deliberately no box for the key above. Colab saves what you type in a
box back into the notebook file, so a key typed there would be written to your
Drive and would travel with the notebook if you ever shared or downloaded it.
Typing it at the prompt keeps it out of the file. If you would rather not
retype it every session, put it in Colab's own secret store instead: the key
icon in the left sidebar, named `OPENALEX_API_KEY`, and this cell will find it.

If your name is common, or your papers are split across more than one OpenAlex
profile, this step tries to stitch them together and will say so.

It writes a file called `benchmark.yaml`. If you have already run this cell and
picked your topics in step 5, running it again would throw those choices away,
so it stops instead. Tick `start_over` if that is what you actually want.

In [ ]:
#@title Your details { display-mode: "form" }
your_email = ""  #@param {type:"string"}
orcid = ""  #@param {type:"string"}
university = ""  #@param {type:"string"}
appointment_start_year = 2019  #@param {type:"integer"}
years_the_clock_was_stopped = 0  #@param {type:"integer"}
start_over = False  #@param {type:"boolean"}

# No form box for the API key on purpose: Colab writes form values back into
# the notebook file, and a secret does not belong in a file you might share.
from tenuretrack.notebook import prompt_for_api_key, run_cli, set_api_key, set_mailto

set_mailto(your_email)
print(set_api_key(prompt_for_api_key()))
print("Looking up", orcid, "at", university)
print()

options = [
    "--orcid", orcid,
    "--institution", university,
    "--start", appointment_start_year,
    "--clock-extension", years_the_clock_was_stopped,
]
run_cli("init", *options, *(["--force"] if start_over else []))

---
## Step 4: Check the topics

This is the one choice that matters most. The cohort is everyone whose research
sits in these topics, so if a topic here is not really your field, the cohort
will include people you would not consider peers.

Read the list below. It comes from your own papers.

In [ ]:
#@title Show what the tool proposed { display-mode: "form" }
from tenuretrack.notebook import describe_config

print(describe_config("benchmark.yaml"))

---
## Step 5: Keep the topics that are yours

Type the numbers of the topics to keep, separated by commas, for example
`1, 2, 4`. Type `all` to keep every one of them.

Look back at step 3, at the line reading "people this topic brings in". That is
how many researchers each topic puts in front of the filters, and it is the
number that decides both how long step 6 takes and how close your cohort is to
your own work. How central a topic is to you does not predict it. A topic
carrying three of your papers can easily bring in five times the people of the
topic carrying thirty.

Topics you drop are written down as deliberately left out, so your report can
say what the subfield does not cover. Four or five topics usually gives a clean
cohort. One topic is too narrow, and six is often wide enough to pull in a
neighboring community.

In [ ]:
#@title Choose your topics { display-mode: "form" }
keep = "all"  #@param {type:"string"}
why_dropped = "outside my subfield"  #@param {type:"string"}

from tenuretrack.notebook import describe_config, keep_topics

keep_topics("benchmark.yaml", keep, note=why_dropped)
print(describe_config("benchmark.yaml"))

---
## Step 6: Build the cohort

This is the long one. Leave the tab open. Progress prints as it goes, and each
line tells you how much of the day's OpenAlex allowance is left.

If it stops partway, whether from a lost connection or from OpenAlex saying the
day's allowance is spent, just run the cell again. Everything already fetched is
saved, so it resumes rather than starting over.

With the free key this should finish in one sitting. Without one it will most
likely stop on the day's allowance and want two or three visits. If you skipped
the key earlier and would rather not have, get one at
[openalex.org/settings/api](https://openalex.org/settings/api) and rerun step 3.

In [ ]:
#@title Build the cohort and compute the norms { display-mode: "form" }
from tenuretrack.notebook import run_cli

run_cli("run")

---
## Step 7: Read your report

The report has four parts:

- **Subfield norms**: what the middle half of your cohort had published by each
  year of the clock (p25, median, p75).
- **You and the cohort** at the same career year. Citations are shown for you
  but deliberately not compared, because your papers have had a few years to
  accumulate citations and your cohort's have had ten or more.
- **The funnel**: how many people were considered, and how many each filter
  removed. This is how you check that the cohort is sensible.
- **Venues**: the journals your subfield actually publishes in, so a
  first-quartile journal means first quartile in your field.

In [ ]:
#@title Show the report { display-mode: "form" }
from pathlib import Path

from IPython.display import Markdown, display

from tenuretrack.notebook import list_results

report = Path("results/report.md")
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print("No report yet. Run step 6 first.")

print("\nFiles written:")
for path in list_results("results"):
    print(" ", path)

---
## Step 8: Download your results

This packages the report, the tables, and the slides into one zip file and
sends it to your computer. Before anything is packaged, the files are scanned
for cohort members' names and IDs. If the scan finds one, nothing is written
and the cell tells you what it found.

In [ ]:
#@title Download the results { display-mode: "form" }
from tenuretrack.notebook import zip_results

bundle = zip_results("results", "tenuretrack-results.zip")
print("Packaged:", bundle)

try:
    from google.colab import files

    files.download(str(bundle))
except ImportError:
    print("Not running in Colab. The zip is next to your results folder.")

---
## Questions this notebook cannot answer

Teaching, mentoring, service, funding, software, datasets, and public
scholarship do not appear in OpenAlex, and they are a large part of the job.
See [docs/beyond-papers.md](https://github.com/sp8rks/tenuretrack/blob/main/docs/beyond-papers.md).

OpenAlex also splits some people across profiles and merges others with
namesakes. The cohort keeps only the people it can identify confidently, which
tilts it slightly toward people with distinctive names.

The full method, including how a career start is estimated and how venue
quartiles are computed, is in
[docs/methods.md](https://github.com/sp8rks/tenuretrack/blob/main/docs/methods.md).

Something look wrong? Open an issue at
[github.com/sp8rks/tenuretrack](https://github.com/sp8rks/tenuretrack/issues).
Please do not paste cohort names into an issue.